# Launch News Trend Analysis

Analyses new-product launch news for a keyword: parses pasted news search results, de-duplicates near-identical articles (TF-IDF cosine similarity + fuzzy matching), groups them into issues, and charts brand / category / USP trends.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


🟦 셀 1 — 설치 + 라이브러리 + 폰트

In [ ]:
# ============================================================
# 0. 라이브러리 설치
# ============================================================

!pip -q install pandas openpyxl matplotlib scikit-learn rapidfuzz


# ============================================================
# 1. 기본 라이브러리
# ============================================================

import re
import os
import math
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rapidfuzz.fuzz import ratio
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, FileLink

warnings.filterwarnings("ignore")


# ============================================================
# 2. 한글 폰트 설정
# ============================================================

!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

🟦 셀 2 — 사용자 입력 + Naver 뉴스 붙여넣기

In [ ]:
# ============================================================
# 3. 사용자 입력
# ============================================================

print("=" * 70)
print("Naver News 시장 트렌드 분석")
print("=" * 70)

search_keyword = input(
    "\n① Naver News 검색어를 입력하세요\n"
    "예: 생리대 출시 / 기저귀 출시 / 물티슈 출시\n"
    "> "
).strip()

category_input = input(
    "\n② 분석할 제품 카테고리를 입력하세요\n"
    "예: 생리대, 팬티형 생리대, 탐폰, 라이너, 기저귀, 요실금 언더웨어, 물티슈\n"
    "> "
).strip()

date_range = input(
    "\n③ 검색 기간을 입력하세요\n"
    "예: 2026.08.22. ~ 2026.09.21.\n"
    "> "
).strip()


# ============================================================
# 4. Naver News 복붙 입력
# ============================================================

print("\n" + "=" * 70)
print("④ Naver News 검색 결과를 전부 붙여넣으세요.")
print("붙여넣기가 끝나면 마지막 줄에 END 를 입력하세요.")
print("=" * 70)

lines = []

while True:
    line = input()

    if line.strip() == "END":
        break

    lines.append(line)

raw_text = "\n".join(lines)

print("\n입력된 문자 수:", len(raw_text))

if len(raw_text) < 100:
    raise ValueError(
        "입력된 데이터가 너무 짧습니다. "
        "Naver News 검색 결과를 전체 복사했는지 확인해주세요."
    )

🟦 셀 3 — Naver 뉴스 파싱

In [ ]:

# ============================================================
# 5. 텍스트 정리
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # Naver UI 문구
    remove_patterns = [
        r"새 창 열림",
        r"프로필 이미지",
        r"의 이미지",
        r"네이버뉴스",
        r"네이버 뉴스",
        r"기사보기",
        r"본문",
        r"댓글",
        r"공유하기",
        r"스크랩",
    ]

    for p in remove_patterns:
        text = re.sub(p, " ", text, flags=re.I)

    # HTML entity
    text = text.replace("&quot;", '"')
    text = text.replace("&amp;", "&")
    text = text.replace("&lt;", "<")
    text = text.replace("&gt;", ">")
    text = text.replace("&#39;", "'")

    # 공백 정리
    text = re.sub(r"\s+", " ", text)

    return text.strip()


cleaned_text = clean_text(raw_text)


# ============================================================
# 6. 상대 날짜 패턴
# ============================================================

TIME_PATTERN = re.compile(
    r"(오늘|어제|\d+\s*시간\s*전|\d+\s*일\s*전|\d+\s*주\s*전|\d+\s*개월\s*전)"
)


def relative_to_date(token):
    """
    Naver 상대 날짜를 오늘 기준 날짜로 변환
    """

    today = datetime.now().date()

    token = token.strip()

    if token == "오늘":
        return today

    if token == "어제":
        return today - timedelta(days=1)

    m = re.search(r"(\d+)\s*시간\s*전", token)

    if m:
        return today

    m = re.search(r"(\d+)\s*일\s*전", token)

    if m:
        return today - timedelta(days=int(m.group(1)))

    m = re.search(r"(\d+)\s*주\s*전", token)

    if m:
        return today - timedelta(days=7 * int(m.group(1)))

    m = re.search(r"(\d+)\s*개월\s*전", token)

    if m:
        return today - timedelta(days=30 * int(m.group(1)))

    return None


time_matches = list(TIME_PATTERN.finditer(cleaned_text))

print("\n발견된 상대 날짜:", len(time_matches))


# ============================================================
# 7. 기사 블록 분리
# ============================================================

article_blocks = []

for i, match in enumerate(time_matches):

    start = match.start()

    if i < len(time_matches) - 1:
        end = time_matches[i + 1].start()
    else:
        end = len(cleaned_text)

    block = cleaned_text[start:end].strip()

    if len(block) >= 20:
        article_blocks.append({
            "relative_date": match.group(1),
            "date": relative_to_date(match.group(1)),
            "raw_block": block
        })


print("기사 블록 후보:", len(article_blocks))


# ============================================================
# 8. 기사 제목 추출용 정리
# ============================================================

def remove_date_prefix(text):
    text = TIME_PATTERN.sub("", text, count=1)
    return text.strip(" |·-:[]")


def split_candidates(text):

    text = remove_date_prefix(text)

    # 구분자 우선
    parts = re.split(
        r"\s{2,}|"
        r"\s+\|\s+|"
        r"\s+[-–—]\s+|"
        r"\s+ㆍ\s+|"
        r"\s+·\s+",
        text
    )

    parts = [
        p.strip(" |·-:[]")
        for p in parts
        if len(p.strip()) >= 8
    ]

    return parts


def clean_title_candidate(title):

    title = title.strip()

    # 지나치게 긴 문장은 제목으로 보기 어려움
    if len(title) > 180:
        title = title[:180]

    # 기사 본문처럼 보이는 표현 제거
    bad_starts = [
        "관계자는",
        "업계에 따르면",
        "이날",
        "이번",
        "오는",
        "지난",
        "한편",
        "특히",
        "또한",
        "제품은",
        "회사 측은",
    ]

    for x in bad_starts:
        if title.startswith(x):
            return ""

    # 너무 긴 문장에 마침표가 여러 개 있으면 본문 가능성
    if title.count(".") >= 2:
        return ""

    return title


# ============================================================
# 9. 기사 제목 / 매체 / 본문 후보 추출
# ============================================================

articles = []

for idx, block in enumerate(article_blocks):

    text = block["raw_block"]

    candidates = split_candidates(text)

    cleaned_candidates = []

    for c in candidates:
        c = clean_title_candidate(c)

        if c:
            cleaned_candidates.append(c)

    if not cleaned_candidates:
        continue

    # 제목 후보 선정
    #
    # 일반적으로 첫 번째 짧은 문장이 제목일 가능성이 높음.
    # 지나치게 짧거나 UI 문구인 것은 제외.
    title = ""

    for c in cleaned_candidates:

        if 15 <= len(c) <= 150:

            # 지나치게 설명형인 경우 후순위
            if c.endswith(("습니다", "합니다", "됩니다")):
                continue

            title = c
            break

    if not title:
        title = cleaned_candidates[0]

    # 전체 텍스트
    full_text = " ".join(cleaned_candidates)

    # 매체 후보
    publisher = ""

    publisher_patterns = [
        r"\b(연합뉴스|뉴시스|뉴스1|머니투데이|한국경제|매일경제|조선비즈|이데일리|"
        r"아시아경제|파이낸셜뉴스|서울경제|전자신문|헤럴드경제|데일리안|"
        r"이뉴스투데이|비즈워치|아이뉴스24|중앙일보|동아일보|조선일보|한겨레|"
        r"경향신문|국민일보|세계일보|한국일보)\b"
    ]

    for p in publisher_patterns:
        m = re.search(p, full_text)

        if m:
            publisher = m.group(1)
            break

    articles.append({
        "기사ID": len(articles) + 1,
        "날짜": block["date"],
        "상대날짜": block["relative_date"],
        "매체": publisher,
        "제목": title,
        "본문후보": full_text,
        "원문블록": text
    })


articles_df = pd.DataFrame(articles)

print("\n초기 기사 수:", len(articles_df))

🟦 셀 4 — 중복 제거 + 브랜드 + 소카테고리

In [ ]:

# ============================================================
# 10. 기사 제목 정규화
# ============================================================

def normalize_title(title):

    if pd.isna(title):
        return ""

    title = str(title).lower()

    # 특수문자 제거
    title = re.sub(r"[^\w가-힣a-zA-Z0-9\s]", " ", title)

    # 불필요한 공통 표현
    stop_patterns = [
        "관련",
        "대해",
        "통해",
        "밝혀",
        "밝혔다",
        "전해",
        "선보여",
        "소개",
        "주목",
        "눈길",
        "관심",
    ]

    for p in stop_patterns:
        title = title.replace(p, " ")

    title = re.sub(r"\s+", " ", title)

    return title.strip()


articles_df["제목정규화"] = articles_df["제목"].apply(normalize_title)


# ============================================================
# 11. 정확 중복 제거
# ============================================================

before_exact = len(articles_df)

articles_df = articles_df.drop_duplicates(
    subset=["제목정규화"]
).reset_index(drop=True)

exact_removed = before_exact - len(articles_df)

print("정확 중복 제거:", exact_removed)
print("현재 기사 수:", len(articles_df))


# ============================================================
# 12. 브랜드 사전
# ============================================================

BRAND_LIST = [
    "유한킴벌리",
    "깨끗한나라",
    "LG유니참",
    "유니참",
    "한국P&G",
    "킴벌리클라크",
    "애경산업",
    "웰크론헬스케어",
    "중원주식회사",
    "모나리자",
    "쌍용C&B",
    "코디",
    "바디프랜드",
    "라이온코리아",
    "다우니",
    "오드리선",
    "라엘",
    "콜만",
    "시크릿데이",
    "좋은느낌",
    "화이트",
    "디펜드",
    "라이프리",
    "하기스",
    "팸퍼스",
    "킨도",
    "마미포코",
    "보솜이",
    "페넬로페",
    "슈퍼대디",
    "베피스",
    "코멧",
    "브라운",
    "물따로",
    "베베숲",
    "앙블랑",
    "릴리유",
    "닥터마틴",
    "바이탈뷰티",
]


def detect_brands(text):

    found = []

    for brand in BRAND_LIST:

        if brand.lower() in text.lower():
            found.append(brand)

    return sorted(set(found))


articles_df["브랜드_다중"] = articles_df["본문후보"].apply(
    detect_brands
)


# ============================================================
# 13. 제품 카테고리 다중 태깅
# ============================================================
#
# 중요:
# 여기서 하나만 고르는 것이 아니라
# 하나의 기사에 여러 소카테고리를 붙임.
#
# 예:
# "여름용 팬티형 기저귀"
# → 기저귀 + 팬티형 + 여름용
#
# 실제 기존 분류표가 있다면 아래 RULES 부분만 수정하면 됨.
# ============================================================

SUBCATEGORY_RULES = {

    "생리대": [
        "생리대",
        "패드",
        "오버나이트",
        "중형",
        "대형",
        "소형",
        "팬티형 생리대",
        "입는 생리대",
        "유기농 생리대",
    ],

    "팬티형 생리대": [
        "팬티형 생리대",
        "입는 생리대",
        "생리팬티",
        "생리용 팬티",
        "웨어러블 생리대",
    ],

    "탐폰": [
        "탐폰",
        "tampon",
    ],

    "라이너": [
        "라이너",
        "팬티라이너",
        "언더웨어 라이너",
    ],

    "아기용 기저귀": [
        "아기 기저귀",
        "유아 기저귀",
        "신생아 기저귀",
        "아기용 기저귀",
        "유아용 기저귀",
        "기저귀",
        "하기스",
        "팸퍼스",
        "킨도",
        "보솜이",
        "페넬로페",
        "마미포코",
        "슈퍼대디",
        "베피스",
    ],

    "성인용 기저귀": [
        "성인 기저귀",
        "성인용 기저귀",
        "성인용품",
        "성인용 언더웨어",
        "성인용 팬티",
    ],

    "요실금 언더웨어": [
        "요실금",
        "요실금 팬티",
        "요실금 언더웨어",
        "흡수 언더웨어",
        "흡수형 언더웨어",
        "흡수팬티",
    ],

    "팬티형 기저귀": [
        "팬티형 기저귀",
        "팬티형",
        "팬티기저귀",
    ],

    "여름용·쿨링": [
        "여름용",
        "여름 기저귀",
        "쿨링",
        "쿨",
        "시원한",
        "냉감",
        "쿨링감",
    ],

    "신생아·이른둥이": [
        "신생아",
        "이른둥이",
        "미숙아",
        "조산아",
        "preemie",
    ],

    "물티슈": [
        "물티슈",
        "아기물티슈",
        "유아물티슈",
        "청소용 물티슈",
        "여성용 물티슈",
        "페미닌 와이프",
        "feminine wipes",
        "wipes",
    ],

    "여성청결제·여성케어": [
        "여성청결제",
        "여성청결",
        "페미닌",
        "feminine care",
        "intimate care",
        "질세정",
        "외음부",
    ],
}


def multi_subcategory_tagging(text):

    text = str(text).lower()

    tags = []

    for category, keywords in SUBCATEGORY_RULES.items():

        for keyword in keywords:

            if keyword.lower() in text:
                tags.append(category)
                break

    return sorted(set(tags))


articles_df["소카테고리_다중"] = articles_df["본문후보"].apply(
    multi_subcategory_tagging
)


🟦 셀 5 — USP 분석

In [ ]:


# ============================================================
# 14. 표준 USP 사전
# ============================================================
#
# 이건 "정답"이 아니라 시장 비교용 표준화 태그.
#
# 새로운 표현은 아래 고정 사전에 없어도
# 별도로 "USP_원문후보"에서 잡아냄.
# ============================================================

USP_RULES = {

    "흡수력·샘방지": [
        "흡수력",
        "흡수",
        "흡수량",
        "샘방지",
        "샘 방지",
        "누수 방지",
        "누수방지",
        "새지 않는",
        "안 새는",
        "안심흡수",
        "흡수 코어",
        "흡수코어",
        "흡수체",
        "leak protection",
        "leakproof",
    ],

    "통기성": [
        "통기성",
        "통풍",
        "에어",
        "숨쉬는",
        "breathable",
        "breathability",
    ],

    "부드러움·촉감": [
        "부드러운",
        "부드러움",
        "소프트",
        "soft",
        "촉감",
        "매끄러운",
        "순한 촉감",
    ],

    "편안함·착용감": [
        "편안",
        "편안한",
        "착용감",
        "핏",
        "밀착",
        "바디핏",
        "몸에 맞는",
        "편하게",
        "comfortable",
        "comfort",
    ],

    "슬림·얇음": [
        "슬림",
        "얇은",
        "초슬림",
        "얇게",
        "두께",
        "barely",
        "ultra thin",
        "thin",
    ],

    "유기농·천연": [
        "유기농",
        "오가닉",
        "organic",
        "천연",
        "natural",
        "순면",
        "유기농 순면",
    ],

    "저자극·민감피부": [
        "저자극",
        "민감",
        "민감성",
        "피부 자극",
        "피부에 순한",
        "dermatest",
        "hypoallergenic",
    ],

    "쿨링·상쾌함": [
        "쿨링",
        "시원",
        "냉감",
        "상쾌",
        "쿨",
        "cooling",
        "fresh",
        "refreshing",
    ],

    "보송함·건조함": [
        "보송",
        "건조",
        "드라이",
        "흡습",
        "습기",
        "dry",
        "dryness",
    ],

    "냄새·탈취": [
        "냄새",
        "악취",
        "탈취",
        "향",
        "무향",
        "odor",
        "odor control",
        "deodor",
    ],

    "활동성·움직임": [
        "활동",
        "움직임",
        "운동",
        "액티브",
        "스포츠",
        "active",
        "mobility",
        "move",
        "movement",
    ],

    "편의성·휴대성": [
        "간편",
        "편리",
        "휴대",
        "여행",
        "외출",
        "온더고",
        "on the go",
        "travel",
        "convenient",
    ],

    "친환경·지속가능": [
        "친환경",
        "지속가능",
        "지속 가능",
        "재활용",
        "생분해",
        "플라스틱 프리",
        "sustainable",
        "eco",
        "biodegradable",
        "recyclable",
    ],

    "피부케어·보습": [
        "보습",
        "피부케어",
        "피부 케어",
        "진정",
        "알로에",
        "moisture",
        "skin care",
        "soothing",
    ],

    "기술·기능": [
        "기술",
        "테크놀로지",
        "technology",
        "코어",
        "흡수체",
        "이중 구조",
        "3중",
        "5중",
        "레이어",
        "구조",
        "system",
    ],

    "마이크로바이옴·pH": [
        "마이크로바이옴",
        "마이크로바이옴",
        "microbiome",
        "pH",
        "ph 밸런스",
        "유산균",
        "probiotic",
    ],
}


def multi_usp_tagging(text):

    text = str(text).lower()

    tags = []

    for usp, keywords in USP_RULES.items():

        for keyword in keywords:

            if keyword.lower() in text:
                tags.append(usp)
                break

    return sorted(set(tags))


articles_df["USP_표준"] = articles_df["본문후보"].apply(
    multi_usp_tagging
)


# ============================================================
# 15. USP 원문 표현 후보 발굴
# ============================================================
#
# 핵심:
# "우리가 미리 정한 USP"에 없는 새로운 표현을 발견하기 위한 영역.
#
# 예:
# 표준 USP = 부드러움·촉감
# 실제 기사 표현 =
# "실크처럼 부드러운"
# "구름처럼 가벼운"
# "피부에 닿는 순간 시원한"
#
# 이런 표현은 별도로 발견.
# ============================================================

USP_STOPWORDS = {
    "제품",
    "제품을",
    "제품이",
    "브랜드",
    "시장",
    "출시",
    "신제품",
    "리뉴얼",
    "기저귀",
    "생리대",
    "물티슈",
    "기사",
    "업계",
    "회사",
    "기업",
    "고객",
    "소비자",
    "판매",
    "매출",
    "이벤트",
    "행사",
    "페어",
    "관계자",
    "대표",
    "사업",
    "사업을",
    "서비스",
    "상품",
    "라인",
    "라인업",
    "시리즈",
    "패키지",
    "브랜드의",
    "제품의",
}


# 흔히 USP 문구에 들어가는 형태
USP_PHRASE_PATTERNS = [

    # ~한 / ~적인 / ~로운
    r"[가-힣]{2,10}(?:한|적인|로운)\s+[가-힣]{1,10}",

    # ~감
    r"[가-힣]{2,12}감",

    # ~력
    r"[가-힣]{2,12}력",

    # ~케어
    r"[가-힣A-Za-z]{2,15}\s*케어",

    # ~핏
    r"[가-힣A-Za-z]{2,15}\s*핏",

    # ~드라이
    r"[가-힣A-Za-z]{2,15}\s*드라이",

    # ~코어
    r"[가-힣A-Za-z]{2,15}\s*코어",

    # 영어 표현
    r"\b(?:ultra\s+thin|barely\s+there|breathable|soft|gentle|fresh|cooling|"
    r"odor\s+control|leak\s+protection|skin\s+friendly|organic|natural|"
    r"sustainable|on\s+the\s+go|active)\b",
]


def extract_raw_usp_candidates(text):

    text = str(text)

    candidates = []

    for pattern in USP_PHRASE_PATTERNS:

        matches = re.findall(pattern, text, flags=re.I)

        for m in matches:

            m = re.sub(r"\s+", " ", m).strip()

            if len(m) < 2:
                continue

            if len(m) > 30:
                continue

            # 단어 단위 stopword
            if m in USP_STOPWORDS:
                continue

            # 숫자만 있는 표현 제거
            if re.fullmatch(r"[\d\s]+", m):
                continue

            candidates.append(m)

    return sorted(set(candidates))


articles_df["USP_원문후보"] = articles_df["본문후보"].apply(
    extract_raw_usp_candidates
)



🟦 셀 6 — 이슈 클러스터링 + 분석 테이블

In [ ]:

# ============================================================
# 16. 이슈 유형 분류
# ============================================================

def classify_issue_type(text):

    text = str(text)

    renewal_words = [
        "리뉴얼",
        "개편",
        "새롭게",
        "업그레이드",
        "업데이트",
        "개선",
        "리브랜딩",
        "재출시",
        "새 단장",
        "새단장",
    ]

    launch_words = [
        "신제품",
        "출시",
        "선보",
        "론칭",
        "런칭",
        "신규",
        "첫 선",
        "첫선",
        "새 제품",
    ]

    for word in renewal_words:

        if word in text:
            return "리뉴얼"

    for word in launch_words:

        if word in text:
            return "신제품"

    return "기타"


articles_df["이슈유형"] = articles_df["제목"].apply(
    classify_issue_type
)


# ============================================================
# 17. 이벤트/제품 핵심어 추출
# ============================================================

EVENT_KEYWORDS = [
    "이노베이션 페어",
    "혁신 페어",
    "신제품",
    "리뉴얼",
    "출시",
    "론칭",
    "런칭",
    "신규",
    "캠페인",
    "공개",
    "전시",
    "박람회",
    "페어",
]


def extract_event_terms(title):

    found = []

    for word in EVENT_KEYWORDS:

        if word in title:
            found.append(word)

    return found


articles_df["이벤트핵심어"] = articles_df["제목"].apply(
    extract_event_terms
)


# ============================================================
# 18. 기사 유사도 계산
# ============================================================

titles = articles_df["제목정규화"].fillna("").tolist()

if len(titles) > 1:

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=1
    )

    tfidf = vectorizer.fit_transform(titles)

    similarity_matrix = cosine_similarity(tfidf)

else:

    similarity_matrix = np.ones((len(titles), len(titles)))


# ============================================================
# 19. 이슈 클러스터링
# ============================================================
#
# 단순히 "제목이 같은가?"만 보지 않음.
#
# 조건:
# - 제목 유사도
# - 같은 브랜드
# - 같은 이벤트 키워드
#
# 를 함께 봄.
# ============================================================

n = len(articles_df)

parent = list(range(n))


def find(x):

    while parent[x] != x:

        parent[x] = parent[parent[x]]
        x = parent[x]

    return x


def union(a, b):

    ra = find(a)
    rb = find(b)

    if ra != rb:
        parent[rb] = ra


for i in range(n):

    for j in range(i + 1, n):

        sim = similarity_matrix[i, j]

        brands_i = set(articles_df.iloc[i]["브랜드_다중"])
        brands_j = set(articles_df.iloc[j]["브랜드_다중"])

        shared_brand = bool(brands_i & brands_j)

        event_i = set(articles_df.iloc[i]["이벤트핵심어"])
        event_j = set(articles_df.iloc[j]["이벤트핵심어"])

        shared_event = bool(event_i & event_j)

        # 매우 유사한 제목
        if sim >= 0.70:

            # 브랜드가 같거나 둘 중 하나에 브랜드가 없는 경우
            if shared_brand or not brands_i or not brands_j:
                union(i, j)

        # 같은 브랜드 + 같은 이벤트
        elif shared_brand and shared_event and sim >= 0.35:

            union(i, j)


# ============================================================
# 20. 클러스터 ID 생성
# ============================================================

root_to_cluster = {}

cluster_ids = []

next_cluster = 1

for i in range(n):

    root = find(i)

    if root not in root_to_cluster:
        root_to_cluster[root] = next_cluster
        next_cluster += 1

    cluster_ids.append(root_to_cluster[root])


articles_df["이슈ID"] = cluster_ids


print("\n생성된 이슈 수:", articles_df["이슈ID"].nunique())


# ============================================================
# 21. 이슈 단위 대표기사 선정
# ============================================================

issue_rows = []

for issue_id, group in articles_df.groupby("이슈ID"):

    group = group.sort_values(
        by="날짜",
        ascending=True
    )

    # 제목 길이가 너무 길지 않으면서
    # 가장 정보량이 많은 제목을 대표기사로 선정
    candidates = group.copy()

    candidates["제목길이"] = candidates["제목"].str.len()

    candidates = candidates[
        (candidates["제목길이"] >= 15)
        & (candidates["제목길이"] <= 120)
    ]

    if len(candidates) == 0:
        candidates = group

    representative = candidates.iloc[0]

    # 브랜드 합치기
    all_brands = []

    for x in group["브랜드_다중"]:
        all_brands.extend(x)

    all_brands = sorted(set(all_brands))

    # 카테고리 합치기
    all_categories = []

    for x in group["소카테고리_다중"]:
        all_categories.extend(x)

    all_categories = sorted(set(all_categories))

    # USP 합치기
    all_usps = []

    for x in group["USP_표준"]:
        all_usps.extend(x)

    all_usps = sorted(set(all_usps))

    # 원문 USP
    all_raw_usps = []

    for x in group["USP_원문후보"]:
        all_raw_usps.extend(x)

    all_raw_usps = sorted(set(all_raw_usps))

    # 이슈 유형
    issue_types = group["이슈유형"].tolist()

    if "신제품" in issue_types:
        issue_type = "신제품"
    elif "리뉴얼" in issue_types:
        issue_type = "리뉴얼"
    else:
        issue_type = "기타"

    issue_rows.append({

        "이슈ID": issue_id,

        "대표이슈": representative["제목"],

        "브랜드": ", ".join(all_brands),

        "제품/제품명": representative["제목"],

        "제품카테고리": ", ".join(all_categories),

        "소카테고리_다중": ", ".join(all_categories),

        "USP_표준": ", ".join(all_usps),

        "USP_원문후보": ", ".join(all_raw_usps),

        "이슈유형": issue_type,

        "기사수": len(group),

        "최초기사일": group["날짜"].min(),

        "최근기사일": group["날짜"].max(),

        "핵심어": ", ".join(
            sorted(
                set(
                    sum(
                        group["이벤트핵심어"].tolist(),
                        []
                    )
                )
            )
        ),

        "대표기사": representative["제목"],

        "관련기사목록": " | ".join(
            group["제목"].tolist()
        ),
    })


issues_df = pd.DataFrame(issue_rows)


# ============================================================
# 22. 소카테고리 빈도
# ============================================================

subcategory_counter = Counter()

for tags in articles_df["소카테고리_다중"]:

    for tag in tags:
        subcategory_counter[tag] += 1


subcategory_frequency = pd.DataFrame(
    subcategory_counter.items(),
    columns=["소카테고리", "기사수"]
).sort_values(
    "기사수",
    ascending=False
).reset_index(drop=True)


# ============================================================
# 23. USP 빈도
# ============================================================

usp_counter = Counter()

for tags in articles_df["USP_표준"]:

    for tag in tags:
        usp_counter[tag] += 1


usp_frequency = pd.DataFrame(
    usp_counter.items(),
    columns=["USP", "기사수"]
).sort_values(
    "기사수",
    ascending=False
).reset_index(drop=True)


# ============================================================
# 24. USP × 시장/카테고리
# ============================================================

usp_market_rows = []

for _, row in articles_df.iterrows():

    categories = row["소카테고리_다중"]
    usps = row["USP_표준"]

    for category in categories:

        for usp in usps:

            usp_market_rows.append({
                "소카테고리": category,
                "USP": usp,
                "기사ID": row["기사ID"]
            })


usp_market_long = pd.DataFrame(usp_market_rows)

if len(usp_market_long) > 0:

    usp_market_pivot = pd.pivot_table(
        usp_market_long,
        index="USP",
        columns="소카테고리",
        values="기사ID",
        aggfunc="nunique",
        fill_value=0
    )

    usp_market_pivot["전체"] = usp_market_pivot.sum(axis=1)

    usp_market_pivot = usp_market_pivot.sort_values(
        "전체",
        ascending=False
    )

else:

    usp_market_pivot = pd.DataFrame()


# ============================================================
# 25. USP × 신제품/리뉴얼
# ============================================================

usp_launch_rows = []

for _, row in articles_df.iterrows():

    issue_type = row["이슈유형"]

    for usp in row["USP_표준"]:

        usp_launch_rows.append({
            "USP": usp,
            "이슈유형": issue_type,
            "기사ID": row["기사ID"]
        })


usp_launch_long = pd.DataFrame(usp_launch_rows)

if len(usp_launch_long) > 0:

    usp_launch_pivot = pd.pivot_table(
        usp_launch_long,
        index="USP",
        columns="이슈유형",
        values="기사ID",
        aggfunc="nunique",
        fill_value=0
    )

    for col in ["신제품", "리뉴얼", "기타"]:

        if col not in usp_launch_pivot.columns:
            usp_launch_pivot[col] = 0

    usp_launch_pivot["전체"] = usp_launch_pivot.sum(axis=1)

    usp_launch_pivot = usp_launch_pivot.sort_values(
        "전체",
        ascending=False
    )

else:

    usp_launch_pivot = pd.DataFrame()


# ============================================================
# 26. 브랜드 × USP
# ============================================================

brand_usp_rows = []

for _, row in articles_df.iterrows():

    for brand in row["브랜드_다중"]:

        for usp in row["USP_표준"]:

            brand_usp_rows.append({
                "브랜드": brand,
                "USP": usp,
                "기사ID": row["기사ID"]
            })


brand_usp_long = pd.DataFrame(brand_usp_rows)

if len(brand_usp_long) > 0:

    brand_usp_pivot = pd.pivot_table(
        brand_usp_long,
        index="브랜드",
        columns="USP",
        values="기사ID",
        aggfunc="nunique",
        fill_value=0
    )

else:

    brand_usp_pivot = pd.DataFrame()


# ============================================================
# 27. 새로운 USP 표현 후보 분석
# ============================================================
#
# 단순 빈도만 보면 "제품", "기저귀" 같은 단어가 올라오므로
# 이미 알려진 카테고리/브랜드/일반어를 제거.
# ============================================================

candidate_counter = Counter()

for candidates in articles_df["USP_원문후보"]:

    for candidate in candidates:

        candidate_counter[candidate] += 1


candidate_rows = []

known_words = set(
    USP_STOPWORDS
    | set(BRAND_LIST)
)

for phrase, count in candidate_counter.items():

    if count < 2:
        continue

    if phrase in known_words:
        continue

    # 너무 일반적인 단어
    if len(phrase) <= 2:
        continue

    candidate_rows.append({
        "USP_원문표현": phrase,
        "기사수": count
    })


new_usp_candidates = pd.DataFrame(
    candidate_rows
)

if len(new_usp_candidates) > 0:

    new_usp_candidates = new_usp_candidates.sort_values(
        "기사수",
        ascending=False
    ).reset_index(drop=True)

else:

    new_usp_candidates = pd.DataFrame(
        columns=["USP_원문표현", "기사수"]
    )


# ============================================================
# 28. 시장별 새로운 USP 후보
# ============================================================

market_raw_usp_rows = []

for _, row in articles_df.iterrows():

    for category in row["소카테고리_다중"]:

        for phrase in row["USP_원문후보"]:

            market_raw_usp_rows.append({
                "소카테고리": category,
                "USP_원문표현": phrase,
                "기사ID": row["기사ID"]
            })


market_raw_usp_df = pd.DataFrame(
    market_raw_usp_rows
)

if len(market_raw_usp_df) > 0:

    market_new_usp_candidates = (
        market_raw_usp_df
        .groupby(
            ["소카테고리", "USP_원문표현"]
        )["기사ID"]
        .nunique()
        .reset_index(name="기사수")
        .sort_values(
            ["소카테고리", "기사수"],
            ascending=[True, False]
        )
    )

else:

    market_new_usp_candidates = pd.DataFrame(
        columns=[
            "소카테고리",
            "USP_원문표현",
            "기사수"
        ]
    )


# ============================================================
# 29. 제품/이슈 분석
# ============================================================

product_analysis = issues_df.copy()

if len(product_analysis) > 0:

    product_analysis["제품/제품명"] = (
        product_analysis["제품/제품명"]
        .fillna("")
        .astype(str)
    )

    product_analysis = product_analysis.sort_values(
        ["기사수", "최근기사일"],
        ascending=[False, False]
    )


# ============================================================
# 30. 브랜드 분석
# ============================================================

brand_rows = []

for _, row in articles_df.iterrows():

    for brand in row["브랜드_다중"]:

        brand_rows.append({
            "브랜드": brand,
            "기사ID": row["기사ID"],
            "이슈ID": row["이슈ID"],
            "이슈유형": row["이슈유형"]
        })


brand_df = pd.DataFrame(brand_rows)

if len(brand_df) > 0:

    brand_analysis = (
        brand_df
        .groupby("브랜드")
        .agg(
            기사수=("기사ID", "nunique"),
            이슈수=("이슈ID", "nunique")
        )
        .reset_index()
        .sort_values(
            "기사수",
            ascending=False
        )
    )

else:

    brand_analysis = pd.DataFrame(
        columns=["브랜드", "기사수", "이슈수"]
    )


🟦 셀 7 — 그래프 생성

In [ ]:

# ============================================================
# 31. 그래프 폴더
# ============================================================

graph_dir = "/content/naver_news_graphs"

os.makedirs(
    graph_dir,
    exist_ok=True
)


# ============================================================
# GRAPH 1
# 전체 시장 USP TOP 15
# ============================================================

graph_paths = []

if len(usp_frequency) > 0:

    plot_df = usp_frequency.head(15).copy()

    plot_df = plot_df.sort_values(
        "기사수",
        ascending=True
    )

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.barh(
        plot_df["USP"],
        plot_df["기사수"]
    )

    ax.set_title(
        f"{search_keyword} 시장 USP TOP 15",
        fontsize=16
    )

    ax.set_xlabel("기사 수")
    ax.set_ylabel("USP")

    plt.tight_layout()

    path = os.path.join(
        graph_dir,
        "01_전체시장_USP_TOP15.png"
    )

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

    graph_paths.append(path)


# ============================================================
# GRAPH 2
# 소카테고리별 USP TOP
# ============================================================

if len(usp_market_pivot) > 0:

    market_columns = [
        c for c in usp_market_pivot.columns
        if c != "전체"
    ]

    for category in market_columns:

        temp = (
            usp_market_pivot[[category]]
            .sort_values(
                category,
                ascending=False
            )
            .head(10)
            .sort_values(
                category,
                ascending=True
            )
        )

        if temp[category].sum() == 0:
            continue

        fig, ax = plt.subplots(
            figsize=(11, 7)
        )

        ax.barh(
            temp.index,
            temp[category]
        )

        ax.set_title(
            f"{category} - USP TOP 10",
            fontsize=15
        )

        ax.set_xlabel("기사 수")
        ax.set_ylabel("USP")

        plt.tight_layout()

        safe_name = re.sub(
            r"[^\w가-힣]+",
            "_",
            category
        )

        path = os.path.join(
            graph_dir,
            f"02_USP_{safe_name}.png"
        )

        plt.savefig(
            path,
            dpi=200,
            bbox_inches="tight"
        )

        plt.show()

        plt.close()

        graph_paths.append(path)


# ============================================================
# GRAPH 3
# 주요 시장 × USP 비교
# ============================================================

if len(usp_market_pivot) > 0:

    top_usps = (
        usp_market_pivot
        .head(10)
        .drop(columns=["전체"], errors="ignore")
    )

    if len(top_usps) > 0:

        fig, ax = plt.subplots(
            figsize=(14, 9)
        )

        top_usps.plot(
            kind="barh",
            ax=ax
        )

        ax.set_title(
            "주요 USP의 시장별 비교",
            fontsize=16
        )

        ax.set_xlabel("기사 수")
        ax.set_ylabel("USP")

        ax.legend(
            title="소카테고리",
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )

        plt.tight_layout()

        path = os.path.join(
            graph_dir,
            "03_주요USP_시장별비교.png"
        )

        plt.savefig(
            path,
            dpi=200,
            bbox_inches="tight"
        )

        plt.show()

        plt.close()

        graph_paths.append(path)


# ============================================================
# GRAPH 4
# 신제품/리뉴얼에서 어떤 USP를 사용하는가
# ============================================================

if len(usp_launch_pivot) > 0:

    top = (
        usp_launch_pivot
        .head(10)
        .index
        .tolist()
    )

    plot = (
        usp_launch_pivot
        .loc[top]
        .copy()
    )

    # 정렬용 전체 합계
    plot["_총합"] = plot.sum(axis=1)

    plot = (
        plot
        .sort_values(
            "_총합",
            ascending=True
        )
        .drop(
            columns="_총합"
        )
    )

    plot = plot[
        [
            c for c in ["신제품", "리뉴얼", "기타"]
            if c in plot.columns
        ]
    ]

    fig, ax = plt.subplots(
        figsize=(13, 9)
    )

    plot.plot(
        kind="barh",
        ax=ax
    )

    ax.set_title(
        "신제품·리뉴얼에서 강조되는 USP",
        fontsize=16
    )

    ax.set_xlabel("기사 수")
    ax.set_ylabel("USP")

    ax.legend(
        title="이슈유형"
    )

    plt.tight_layout()

    path = os.path.join(
        graph_dir,
        "04_신제품_리뉴얼_USP.png"
    )

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

    graph_paths.append(path)


# ============================================================
# GRAPH 5
# 새로운 USP 원문 표현 TOP 20
# ============================================================

if len(new_usp_candidates) > 0:

    plot_df = (
        new_usp_candidates
        .head(20)
        .sort_values(
            "기사수",
            ascending=True
        )
    )

    fig, ax = plt.subplots(
        figsize=(12, 9)
    )

    ax.barh(
        plot_df["USP_원문표현"],
        plot_df["기사수"]
    )

    ax.set_title(
        "새롭게 등장하는 USP 원문 표현 후보 TOP 20",
        fontsize=16
    )

    ax.set_xlabel("기사 수")
    ax.set_ylabel("원문 표현")

    plt.tight_layout()

    path = os.path.join(
        graph_dir,
        "05_신규USP_원문후보_TOP20.png"
    )

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

    graph_paths.append(path)


# ============================================================
# 32. 시장별 USP 강도 요약
# ============================================================
#
# 각 소카테고리에서 어떤 USP가 상대적으로 많이 등장하는지
# 한눈에 볼 수 있도록 정리.
# ============================================================

market_usp_summary_rows = []

if len(usp_market_pivot) > 0:

    for category in [
        c for c in usp_market_pivot.columns
        if c != "전체"
    ]:

        temp = (
            usp_market_pivot[category]
            .sort_values(
                ascending=False
            )
        )

        temp = temp[temp > 0].head(10)

        for rank, (usp, count) in enumerate(
            temp.items(),
            start=1
        ):

            market_usp_summary_rows.append({

                "소카테고리": category,

                "순위": rank,

                "USP": usp,

                "기사수": int(count)
            })


market_usp_summary = pd.DataFrame(
    market_usp_summary_rows
)


🟦 셀 8 — 결과 확인 + Excel 저장 + 다운로드

In [ ]:

# ============================================================
# 33. 기사 데이터 보기
# ============================================================

print("=" * 70)
print("기사 데이터 보기")
print("=" * 70)

# ------------------------------------------------------------
# articles_df에 이슈 정보가 없으므로
# issues_df에서 이슈 정보를 기사ID 기준으로 연결
# ------------------------------------------------------------

if "이슈ID" not in articles_df.columns:

    if (
        "기사ID" in issues_df.columns
        and "이슈ID" in issues_df.columns
    ):

        issue_article_map = (
            issues_df[
                ["기사ID", "이슈ID"]
            ]
            .drop_duplicates(
                subset=["기사ID"]
            )
        )

        articles_df = articles_df.merge(
            issue_article_map,
            on="기사ID",
            how="left"
        )

        print("✓ 이슈ID를 issues_df에서 기사ID 기준으로 연결했습니다.")

    else:

        print(
            "⚠️ issues_df에 기사ID 또는 이슈ID가 없습니다."
        )


# ------------------------------------------------------------
# 이슈유형도 issues_df에서 연결
# ------------------------------------------------------------

if "이슈유형" not in articles_df.columns:

    if (
        "기사ID" in issues_df.columns
        and "이슈유형" in issues_df.columns
    ):

        issue_type_map = (
            issues_df[
                ["기사ID", "이슈유형"]
            ]
            .drop_duplicates(
                subset=["기사ID"]
            )
        )

        articles_df = articles_df.merge(
            issue_type_map,
            on="기사ID",
            how="left"
        )

        print("✓ 이슈유형을 issues_df에서 기사ID 기준으로 연결했습니다.")

    else:

        print(
            "⚠️ issues_df에 기사ID 또는 이슈유형이 없습니다."
        )


# ------------------------------------------------------------
# 실제 존재하는 컬럼만 확인
# ------------------------------------------------------------

show_columns = [
    "기사ID",
    "날짜",
    "매체",
    "제목",
    "브랜드_다중",
    "소카테고리_다중",
    "USP_표준",
    "USP_원문후보",
    "이슈유형",
    "이슈ID"
]

missing_columns = [
    col
    for col in show_columns
    if col not in articles_df.columns
]

if missing_columns:

    print("\n⚠️ 아직 없는 컬럼:")
    print(missing_columns)

    # 없는 컬럼은 제외하고 표시
    available_columns = [
        col
        for col in show_columns
        if col in articles_df.columns
    ]

else:

    available_columns = show_columns


display(
    articles_df[
        available_columns
    ].head(20)
)

print(
    f"\n현재 기사 데이터: {len(articles_df):,}건"
)
print(
    f"현재 기사 데이터 컬럼 수: {len(articles_df.columns):,}개"
)
# ============================================================
# 34. 이슈 분석 미리보기
# ============================================================

print("\n" + "=" * 70)
print("이슈 분석 TOP 20")
print("=" * 70)

display(
    issues_df[
        [
            "이슈ID",
            "대표이슈",
            "브랜드",
            "제품카테고리",
            "USP_표준",
            "이슈유형",
            "기사수",
            "최초기사일",
            "최근기사일"
        ]
    ]
    .sort_values(
        "기사수",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 35. USP TOP
# ============================================================

print("\n" + "=" * 70)
print("USP TOP 20")
print("=" * 70)

display(
    usp_frequency.head(20)
)


# ============================================================
# 36. 새로운 USP 후보
# ============================================================

print("\n" + "=" * 70)
print("새로운 USP 원문 표현 후보")
print("=" * 70)

display(
    new_usp_candidates.head(30)
)



In [ ]:
# ============================================================
# ⑧. 통합 검수 v2
# ------------------------------------------------------------
# 핵심 원칙
# ① 검색어가 들어갔다고 관련 기사로 보지 않음
# ② 실제 대상 제품의 출시/리뉴얼 기사인지 확인
# ③ 브랜드 홍보/조회수/기업 일반기사는 제외
# ④ 타 카테고리 제품 출시도 제외
# ⑤ 제목이 본문처럼 길어도 오류로 보지 않음
# ⑥ 같은 제품/같은 출시 이슈의 여러 기사는 하나의 이슈로 묶을 후보로 표시
# ============================================================

import re
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

print("=" * 80)
print("⑧ 통합 검수 v2 시작")
print("=" * 80)


# ============================================================
# 1. 기본 데이터
# ============================================================

# ============================================================
# 1. 기본 데이터 초기화
# ------------------------------------------------------------
# 이전 ⑧ 실행에서 articles_df에 추가된 검수용 컬럼 제거
# ============================================================

check_articles = articles_df.copy()
check_issues = issues_df.copy()


# 이전 검수 실행에서 생성된 컬럼
OLD_VALIDATION_COLS = [
    "관련성_판정",
    "관련성_사유",
    "관련카테고리",
    "관련키워드",
    "이슈유형_검수",
    "이슈유형_검수결과",
    "제목검수",
    "중복검수",
    "이슈ID검수",
    "이슈유형_근거검수",
    "최종검수",
    "무관사유",
    "_제목",
    "_본문",
    "_전체텍스트",
    "_검색텍스트",
    "_제목정규화",
]


# 중복 컬럼까지 안전하게 제거
check_articles = check_articles.loc[
    :,
    ~check_articles.columns.duplicated()
].copy()


# 이전 검수 결과 제거
check_articles = check_articles.drop(
    columns=[
        col
        for col in OLD_VALIDATION_COLS
        if col in check_articles.columns
    ],
    errors="ignore"
)


# ============================================================
# 기본 컬럼 확보
# ============================================================

required_article_cols = [
    "기사ID",
    "날짜",
    "매체",
    "제목",
    "본문후보",
    "브랜드_다중",
    "소카테고리_다중",
    "USP_표준",
    "USP_원문후보",
    "이슈유형",
    "이슈ID"
]

for col in required_article_cols:
    if col not in check_articles.columns:
        check_articles[col] = ""


required_issue_cols = [
    "이슈ID",
    "대표이슈",
    "브랜드",
    "제품카테고리",
    "USP_표준",
    "이슈유형",
    "기사수",
    "최초기사일",
    "최근기사일"
]

for col in required_issue_cols:
    if col not in check_issues.columns:
        check_issues[col] = ""


# ============================================================
# 최종 안전장치
# ============================================================

check_articles = check_articles.loc[
    :,
    ~check_articles.columns.duplicated()
].copy()

print("검수용 기사 데이터:", check_articles.shape)
print("중복 컬럼:", check_articles.columns[
    check_articles.columns.duplicated()
].tolist())

required_article_cols = [
    "기사ID", "날짜", "매체", "제목", "본문후보",
    "브랜드_다중", "소카테고리_다중",
    "USP_표준", "USP_원문후보",
    "이슈유형", "이슈ID"
]

for col in required_article_cols:
    if col not in check_articles.columns:
        check_articles[col] = ""

required_issue_cols = [
    "이슈ID", "대표이슈", "브랜드",
    "제품카테고리", "USP_표준",
    "이슈유형", "기사수",
    "최초기사일", "최근기사일"
]

for col in required_issue_cols:
    if col not in check_issues.columns:
        check_issues[col] = ""


# ============================================================
# 2. 텍스트 정리
# ============================================================

def norm_text(x):
    if pd.isna(x):
        return ""

    x = str(x).lower()

    x = re.sub(r"<[^>]+>", " ", x)

    html_entities = {
        "&quot;": '"',
        "&amp;": "&",
        "&lt;": "<",
        "&gt;": ">",
        "&#39;": "'"
    }

    for k, v in html_entities.items():
        x = x.replace(k, v)

    ui_patterns = [
        "새 창 열림",
        "프로필 이미지",
        "의 이미지",
        "네이버뉴스",
        "네이버 뉴스",
        "기사보기",
        "댓글",
        "공유하기",
        "스크랩"
    ]

    for p in ui_patterns:
        x = x.replace(p.lower(), " ")

    x = re.sub(r"\s+", " ", x)

    return x.strip()


check_articles["_제목"] = (
    check_articles["제목"]
    .fillna("")
    .astype(str)
    .map(norm_text)
)

check_articles["_본문"] = (
    check_articles["본문후보"]
    .fillna("")
    .astype(str)
    .map(norm_text)
)

check_articles["_전체텍스트"] = (
    check_articles["_제목"] + " " +
    check_articles["_본문"]
).str.strip()


# ============================================================
# 3. 핵심 카테고리
# ------------------------------------------------------------
# "물티슈"라는 단어만 있다고 관련 제품으로 판단하지 않음.
# 실제 제품/출시 맥락을 같이 확인.
# ============================================================

TARGET_CATEGORIES = {

    "생리대": [
        "생리대",
        "입는 생리대",
        "팬티형 생리대",
        "생리 패드",
        "생리용 패드",
        "menstrual pad",
        "period pad"
    ],

    "생리용 언더웨어": [
        "생리팬티",
        "생리용 팬티",
        "생리용 언더웨어",
        "입는 생리대",
        "period underwear",
        "period panties",
        "wearable pad",
        "웨어러블 패드"
    ],

    "물티슈": [
        "물티슈",
        "여성청결티슈",
        "여성 청결 티슈",
        "청결티슈",
        "flushable wipes"
    ],

    "여성위생": [
        "여성위생용품",
        "여성 위생용품",
        "여성용품",
        "여성청결",
        "페미닌 케어",
        "feminine care",
        "intimate care"
    ]
}


# ============================================================
# 4. 명백한 타 카테고리
# ============================================================

OTHER_PRODUCT_CATEGORIES = [
    "휴지걸이",
    "선반형 휴지걸이",
    "안전바",
    "욕실용품",
    "욕실 제품",
    "화장지",
    "두루마리",
    "키친타월",
    "주방용품",
    "샤워용품",
    "세면대",
    "변기",
]


# ============================================================
# 5. 출시 / 리뉴얼 표현
# ============================================================

NEW_PRODUCT_PATTERNS = [
    r"신제품",
    r"제품을 출시",
    r"제품 출시",
    r"출시했다",
    r"출시하며",
    r"출시하고",
    r"출시해",
    r"출시된",
    r"정식 출시",
    r"새롭게 출시",
    r"선보였다",
    r"선보여",
    r"선보인다",
    r"선보인",
    r"신제품을 선보",
    r"새 제품을 선보",
    r"론칭",
    r"런칭"
]

RENEWAL_PATTERNS = [
    r"리뉴얼",
    r"새단장",
    r"업그레이드",
    r"재출시",
    r"제품을 개선",
    r"제품 개선",
    r"리뉴얼 출시"
]


# ============================================================
# 6. 제품 출시와 무관한 기사 문맥
# ============================================================

NON_LAUNCH_PATTERNS = [
    r"브랜드 영상",
    r"영상.*100만",
    r"100만.*뷰",
    r"조회수",
    r"캠페인",
    r"광고",
    r"마케팅",
    r"브랜드.*활동",
    r"소비자.*소통",
    r"매출",
    r"영업이익",
    r"실적",
    r"수상",
    r"기부",
    r"사회공헌",
    r"채용",
    r"인사",
    r"대표.*선임"
]


def regex_hits(text, patterns):
    hits = []

    for pattern in patterns:
        if re.search(pattern, text):
            hits.append(pattern)

    return hits


# ============================================================
# 7. 관련 카테고리 탐색
# ============================================================

def find_target_categories(text):

    found = []

    for category, keywords in TARGET_CATEGORIES.items():

        if any(k.lower() in text for k in keywords):
            found.append(category)

    return found


# ============================================================
# 8. 관련성 판정
# ------------------------------------------------------------
# 중요:
# "카테고리 키워드 + 출시/리뉴얼"이 있어야 자동 관련.
#
# 단순히 본문에 물티슈/생리대가 언급된 것만으로는 관련 처리하지 않음.
# ============================================================

def relevance_check(row):

    title = row["_제목"]
    text = row["_전체텍스트"]

    target_categories = find_target_categories(text)

    new_hits = regex_hits(text, NEW_PRODUCT_PATTERNS)
    renewal_hits = regex_hits(text, RENEWAL_PATTERNS)
    non_launch_hits = regex_hits(text, NON_LAUNCH_PATTERNS)

    other_category_hits = [
        x for x in OTHER_PRODUCT_CATEGORIES
        if x in text
    ]


    # --------------------------------------------------------
    # A. 명백한 타 카테고리 제품
    # --------------------------------------------------------

    # 제목에 타 카테고리 제품 출시가 명확하면 우선 제외
    title_other = [
        x for x in OTHER_PRODUCT_CATEGORIES
        if x in title
    ]

    if title_other:

        # 타 제품이 제목에 있고
        # 대상 제품이 제목에 없으면 무관
        title_target = find_target_categories(title)

        if not title_target:

            return pd.Series({
                "관련성_판정": "무관",
                "관련성_사유": "타 카테고리 제품 출시 기사",
                "관련카테고리": "",
                "관련키워드": ", ".join(title_other)
            })


    # --------------------------------------------------------
    # B. 브랜드 영상 / 조회수 / 캠페인
    # --------------------------------------------------------

    if non_launch_hits:

        # 제목 자체가 브랜드 콘텐츠라면 무관
        if any(
            x in title
            for x in [
                "브랜드 영상",
                "100만 뷰",
                "조회수",
                "캠페인",
                "광고"
            ]
        ):

            return pd.Series({
                "관련성_판정": "무관",
                "관련성_사유": "제품 출시가 아닌 브랜드/마케팅 기사",
                "관련카테고리": "",
                "관련키워드": ", ".join(non_launch_hits)
            })


    # --------------------------------------------------------
    # C. 대상 카테고리가 제목에 직접 등장
    # --------------------------------------------------------

    title_categories = find_target_categories(title)

    if title_categories:

        if new_hits or renewal_hits:

            return pd.Series({
                "관련성_판정": "관련",
                "관련성_사유": "대상 제품의 출시/리뉴얼 기사",
                "관련카테고리": ", ".join(title_categories),
                "관련키워드": ", ".join(
                    title_categories + new_hits + renewal_hits
                )
            })


    # --------------------------------------------------------
    # D. 제목에 제품이 없더라도 본문에서 명확한 제품 출시
    # --------------------------------------------------------

    if target_categories and (new_hits or renewal_hits):

        # 타 제품만 출시되고 대상 제품은 부수적으로 언급되는 경우
        if other_category_hits and not title_categories:

            # 대상 카테고리가 출시 대상인지 확인
            # 단순히 "물티슈 같은 물건" 같은 문맥이면 제외
            product_launch_phrases = [
                "물티슈를 출시",
                "물티슈 출시",
                "물티슈를 선보",
                "생리대를 출시",
                "생리대 출시",
                "생리대를 선보",
                "언더웨어를 출시",
                "언더웨어 출시",
                "생리용품을 출시",
                "여성용품을 출시"
            ]

            if not any(
                phrase in text
                for phrase in product_launch_phrases
            ):

                return pd.Series({
                    "관련성_판정": "무관",
                    "관련성_사유": "대상 카테고리가 본문에 단순 언급됨",
                    "관련카테고리": ", ".join(target_categories),
                    "관련키워드": ", ".join(target_categories)
                })


        return pd.Series({
            "관련성_판정": "관련",
            "관련성_사유": "본문에서 대상 제품의 출시/리뉴얼 확인",
            "관련카테고리": ", ".join(target_categories),
            "관련키워드": ", ".join(
                target_categories + new_hits + renewal_hits
            )
        })


    # --------------------------------------------------------
    # E. 제품 카테고리는 있지만 출시 여부 불명확
    # --------------------------------------------------------

    if target_categories:

        return pd.Series({
            "관련성_판정": "재검토",
            "관련성_사유": "대상 제품은 언급되나 출시/리뉴얼 근거 불명확",
            "관련카테고리": ", ".join(target_categories),
            "관련키워드": ", ".join(target_categories)
        })


    # --------------------------------------------------------
    # F. 완전 무관
    # --------------------------------------------------------

    return pd.Series({
        "관련성_판정": "무관",
        "관련성_사유": "대상 시장/제품과 직접적인 관련 없음",
        "관련카테고리": "",
        "관련키워드": ""
    })


relevance_result = check_articles.apply(
    relevance_check,
    axis=1
)

check_articles = pd.concat(
    [check_articles, relevance_result],
    axis=1
)


# ============================================================
# 9. 신제품 / 리뉴얼 판정
# ------------------------------------------------------------
# 관련 기사로 판정된 기사에 대해서만 적용
# ============================================================

def classify_issue_type(row):

    if row["관련성_판정"] != "관련":
        return "기타"

    text = row["_전체텍스트"]

    renewal_hits = regex_hits(
        text,
        RENEWAL_PATTERNS
    )

    new_hits = regex_hits(
        text,
        NEW_PRODUCT_PATTERNS
    )

    if renewal_hits:
        return "리뉴얼"

    if new_hits:
        return "신제품"

    return "기타"


check_articles["이슈유형_검수"] = check_articles.apply(
    classify_issue_type,
    axis=1
)


# ============================================================
# 10. 기존 이슈유형과 비교
# ============================================================

def type_consistency(row):

    existing = str(
        row.get("이슈유형", "")
    ).strip()

    checked = str(
        row.get("이슈유형_검수", "")
    ).strip()

    if not existing or existing.lower() == "nan":
        return "기존값없음"

    if existing == checked:
        return "일치"

    return "불일치"


check_articles["이슈유형_검수결과"] = check_articles.apply(
    type_consistency,
    axis=1
)


# ============================================================
# 11. 제목 검수
# ------------------------------------------------------------
# 길이 / 본문 포함 여부는 오류가 아님
# ============================================================

def title_check(title):

    title = str(title or "").strip()

    if not title or title.lower() == "nan":
        return "제목없음"

    return "정상"


check_articles["제목검수"] = (
    check_articles["제목"]
    .map(title_check)
)


# ============================================================
# 12. 중복 검수
# ============================================================

def normalize_title(title):

    x = norm_text(title)

    x = re.sub(
        r"[^0-9a-z가-힣 ]",
        "",
        x
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


check_articles["_제목정규화"] = (
    check_articles["제목"]
    .map(normalize_title)
)

check_articles["중복검수"] = "정상"


# 정확 중복
duplicate_mask = (
    check_articles["_제목정규화"].duplicated(
        keep=False
    )
    &
    (check_articles["_제목정규화"] != "")
)

check_articles.loc[
    duplicate_mask,
    "중복검수"
] = "정확중복"


# 유사 중복
near_duplicate_pairs = []

for i in range(len(check_articles)):

    for j in range(i + 1, len(check_articles)):

        a = check_articles.iloc[i]
        b = check_articles.iloc[j]

        title_a = a["_제목정규화"]
        title_b = b["_제목정규화"]

        if not title_a or not title_b:
            continue

        if len(title_a) < 10 or len(title_b) < 10:
            continue

        similarity = SequenceMatcher(
            None,
            title_a,
            title_b
        ).ratio()

        if similarity >= 0.82:

            near_duplicate_pairs.append({
                "기사ID_1": a["기사ID"],
                "기사ID_2": b["기사ID"],
                "제목_1": a["제목"],
                "제목_2": b["제목"],
                "유사도": round(similarity, 3)
            })

            check_articles.loc[
                check_articles["기사ID"].isin(
                    [a["기사ID"], b["기사ID"]]
                ),
                "중복검수"
            ] = "유사중복"


near_duplicate_df = pd.DataFrame(
    near_duplicate_pairs
)


# ============================================================
# 13. 이슈ID 검수
# ============================================================

valid_issue_ids = set(
    check_issues["이슈ID"]
    .dropna()
    .astype(str)
    .str.strip()
)


def issue_id_check(value):

    if pd.isna(value):
        return "이슈ID없음"

    value = str(value).strip()

    if not value:
        return "이슈ID없음"

    if value not in valid_issue_ids:
        return "존재하지않는이슈ID"

    return "정상"


check_articles["이슈ID검수"] = (
    check_articles["이슈ID"]
    .map(issue_id_check)
)


# ============================================================
# 14. 이슈유형 근거 검수
# ============================================================

def evidence_check(row):

    existing = str(
        row.get("이슈유형", "")
    ).strip()

    checked = row["이슈유형_검수"]

    if not existing or existing.lower() == "nan":
        return "기존 이슈유형 없음"

    if existing == "리뉴얼" and checked != "리뉴얼":
        return "리뉴얼 근거 없음"

    if existing == "신제품" and checked != "신제품":
        return "신제품 근거 없음"

    return "정상"


check_articles["이슈유형_근거검수"] = (
    check_articles.apply(
        evidence_check,
        axis=1
    )
)


# ============================================================
# 15. 동일 이슈 후보
# ------------------------------------------------------------
# 같은 브랜드 + 같은 날짜 + 제목 유사성
# ------------------------------------------------------------

same_issue_candidates = []

for i in range(len(check_articles)):

    for j in range(i + 1, len(check_articles)):

        a = check_articles.iloc[i]
        b = check_articles.iloc[j]

        brand_a = str(
            a.get("브랜드_다중", "")
        ).strip()

        brand_b = str(
            b.get("브랜드_다중", "")
        ).strip()

        if not brand_a or not brand_b:
            continue

        if brand_a.lower() == "nan":
            continue

        if brand_b.lower() == "nan":
            continue

        brands_a = set(
            x.strip()
            for x in re.split(
                r"[,|/]",
                brand_a
            )
            if x.strip()
        )

        brands_b = set(
            x.strip()
            for x in re.split(
                r"[,|/]",
                brand_b
            )
            if x.strip()
        )

        common_brand = brands_a.intersection(
            brands_b
        )

        if not common_brand:
            continue

        date_a = str(a["날짜"])
        date_b = str(b["날짜"])

        title_a = normalize_title(a["제목"])
        title_b = normalize_title(b["제목"])

        similarity = SequenceMatcher(
            None,
            title_a,
            title_b
        ).ratio()

        if (
            date_a == date_b
            or similarity >= 0.60
        ):

            same_issue_candidates.append({

                "기사ID_1": a["기사ID"],
                "기사ID_2": b["기사ID"],

                "브랜드": ", ".join(
                    common_brand
                ),

                "날짜_1": date_a,
                "날짜_2": date_b,

                "제목_1": a["제목"],
                "제목_2": b["제목"],

                "유사도": round(
                    similarity,
                    3
                ),

                "현재이슈ID_1": a["이슈ID"],
                "현재이슈ID_2": b["이슈ID"]
            })


same_issue_candidates_df = pd.DataFrame(
    same_issue_candidates
)


# ============================================================
# 16. 기사별 최종 검수
# ============================================================

def final_judgement(row):

    issues = []

    if row["관련성_판정"] == "무관":
        issues.append("무관기사")

    elif row["관련성_판정"] == "재검토":
        issues.append("관련성재검토")

    if row["중복검수"] != "정상":
        issues.append(
            row["중복검수"]
        )

    if row["이슈ID검수"] != "정상":
        issues.append(
            row["이슈ID검수"]
        )

    if row["이슈유형_근거검수"] not in [
        "정상",
        "기존 이슈유형 없음"
    ]:
        issues.append(
            row["이슈유형_근거검수"]
        )

    if row["제목검수"] == "제목없음":
        issues.append("제목없음")

    if not issues:
        return "통과"

    return " / ".join(
        dict.fromkeys(issues)
    )


check_articles["최종검수"] = (
    check_articles.apply(
        final_judgement,
        axis=1
    )
)


# ============================================================
# 17. 결과 출력
# ============================================================

print("\n" + "=" * 80)
print("⑧-A. 전체 흐름")
print("=" * 80)

print(
    f"상대 날짜 발견 수 : {len(time_matches):,}"
)

print(
    f"기사 블록 후보    : {len(article_blocks):,}"
)

print(
    f"현재 파싱 기사    : {len(check_articles):,}"
)

print(
    "\n※ 블록 → 기사 감소율은 기사 누락률로 계산하지 않습니다."
)


# ============================================================
# 18. 관련성
# ============================================================

print("\n" + "=" * 80)
print("⑧-B. 관련성 검수")
print("=" * 80)

print(
    check_articles["관련성_판정"]
    .value_counts(
        dropna=False
    )
)

relevance_review = check_articles[
    check_articles["관련성_판정"]
    .isin(
        ["무관", "재검토"]
    )
][
    [
        "기사ID",
        "날짜",
        "제목",
        "관련성_판정",
        "관련성_사유",
        "관련카테고리",
        "관련키워드"
    ]
]

if len(relevance_review):

    print(
        "\n[관련성 확인 대상]"
    )

    display(
        relevance_review
    )


# ============================================================
# 19. 신제품 / 리뉴얼
# ============================================================

print("\n" + "=" * 80)
print("⑧-C. 신제품 / 리뉴얼")
print("=" * 80)

print(
    check_articles[
        "이슈유형_검수"
    ].value_counts(
        dropna=False
    )
)

type_review = check_articles[
    check_articles[
        "이슈유형_근거검수"
    ].isin(
        [
            "리뉴얼 근거 없음",
            "신제품 근거 없음"
        ]
    )
][
    [
        "기사ID",
        "날짜",
        "제목",
        "이슈유형",
        "이슈유형_검수",
        "이슈유형_근거검수"
    ]
]

if len(type_review):

    print(
        "\n[이슈유형 확인 대상]"
    )

    display(
        type_review
    )


# ============================================================
# 20. 중복
# ============================================================

print("\n" + "=" * 80)
print("⑧-D. 중복 검수")
print("=" * 80)

print(
    check_articles[
        "중복검수"
    ].value_counts(
        dropna=False
    )
)

if len(near_duplicate_df):

    print(
        "\n[유사 중복 후보]"
    )

    display(
        near_duplicate_df
    )


# ============================================================
# 21. 이슈ID
# ============================================================

print("\n" + "=" * 80)
print("⑧-E. 기사 ↔ 이슈 연결")
print("=" * 80)

print(
    check_articles[
        "이슈ID검수"
    ].value_counts(
        dropna=False
    )
)

issue_link_review = check_articles[
    check_articles[
        "이슈ID검수"
    ] != "정상"
][
    [
        "기사ID",
        "날짜",
        "제목",
        "이슈ID",
        "이슈ID검수"
    ]
]

if len(issue_link_review):

    display(
        issue_link_review
    )


# ============================================================
# 22. 동일 이슈 후보
# ============================================================

print("\n" + "=" * 80)
print("⑧-F. 동일 이슈 가능성")
print("=" * 80)

if len(same_issue_candidates_df):

    display(
        same_issue_candidates_df
    )

else:

    print(
        "현재 자동 탐지된 동일 이슈 후보 없음"
    )


# ============================================================
# 23. 최종 검수
# ============================================================

print("\n" + "=" * 80)
print("⑧-G. 최종 검수 필요 기사")
print("=" * 80)

final_review = check_articles[
    check_articles[
        "최종검수"
    ] != "통과"
][
    [
        "기사ID",
        "날짜",
        "매체",
        "제목",
        "관련성_판정",
        "관련성_사유",
        "이슈유형",
        "이슈유형_검수",
        "이슈ID",
        "중복검수",
        "이슈ID검수",
        "이슈유형_근거검수",
        "최종검수"
    ]
]

if len(final_review):

    display(
        final_review
    )

else:

    print(
        "최종 검수 필요 기사 없음"
    )


# ============================================================
# 24. 핵심 요약
# ============================================================

print("\n" + "=" * 80)
print("⑧-H. 핵심 요약")
print("=" * 80)

total = len(check_articles)

related = (
    check_articles[
        "관련성_판정"
    ] == "관련"
).sum()

review_related = (
    check_articles[
        "관련성_판정"
    ] == "재검토"
).sum()

unrelated = (
    check_articles[
        "관련성_판정"
    ] == "무관"
).sum()

duplicate_count = (
    check_articles[
        "중복검수"
    ] != "정상"
).sum()

issue_error_count = (
    check_articles[
        "이슈ID검수"
    ] != "정상"
).sum()

print(
    f"현재 파싱 기사       : {total:,}"
)

print(
    f"관련 기사             : {related:,}"
)

print(
    f"관련성 재검토         : {review_related:,}"
)

print(
    f"무관 기사             : {unrelated:,}"
)

print(
    f"중복/유사중복         : {duplicate_count:,}"
)

print(
    f"이슈ID 연결 오류      : {issue_error_count:,}"
)

print(
    "\n※ 제목이 본문처럼 길어도 그 자체로 오류 처리하지 않습니다."
)

print(
    "※ '출시'라는 단어가 있다고 신제품으로 자동 인정하지 않습니다."
)

print(
    "※ 타 카테고리 제품 출시와 브랜드 홍보 기사는 제외 대상으로 분리합니다."
)


# ============================================================
# 25. articles_df에 검수 결과 반영
# ============================================================

VALIDATION_COLS = [
    "관련성_판정",
    "관련성_사유",
    "관련카테고리",
    "관련키워드",
    "이슈유형_검수",
    "이슈유형_검수결과",
    "제목검수",
    "중복검수",
    "이슈ID검수",
    "이슈유형_근거검수",
    "최종검수"
]

for col in VALIDATION_COLS:

    articles_df[col] = (
        check_articles[col].values
    )


# 내부용 컬럼 제거
for col in [
    "_제목",
    "_본문",
    "_전체텍스트",
    "_제목정규화"
]:

    if col in articles_df.columns:

        articles_df.drop(
            columns=[col],
            inplace=True
        )


print("\n⑧ 통합 검수 v2 완료")

In [ ]:
# ============================================================
# ⑨ 최종 Excel 저장
# ============================================================

print("=" * 80)
print("⑨ 최종 Excel 저장")
print("=" * 80)

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import CellIsRule


# ------------------------------------------------------------
# 파일명
# ------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

keyword_safe = re.sub(
    r'[\\/:*?"<>|]',
    "_",
    str(search_keyword)
)

output_file = (
    f"네이버뉴스_시장트렌드_"
    f"{keyword_safe}_"
    f"{timestamp}.xlsx"
)


# ------------------------------------------------------------
# 저장할 데이터 준비
# ------------------------------------------------------------

excel_sheets = {}


# 1. 시장요약
if "market_summary" in globals():
    excel_sheets["시장요약"] = market_summary

elif "summary_df" in globals():
    excel_sheets["시장요약"] = summary_df

else:
    excel_sheets["시장요약"] = review_summary


# 2. 전체기사
excel_sheets["전체기사"] = articles_df.copy()


# 3. 전체이슈
if "issue_df" in globals():
    excel_sheets["전체이슈"] = issue_df.copy()

elif "issue_group_check" in globals():
    excel_sheets["전체이슈"] = issue_group_check.copy()


# 4. 기존 분석 데이터
optional_tables = {
    "소카테고리빈도": "subcategory_freq",
    "USP빈도": "usp_freq",
    "전체USP": "usp_market_df",
    "USP_시장비교": "usp_market_compare",
    "USP_신제품리뉴얼": "usp_launch_df",
    "브랜드_USP": "brand_usp_df",
    "신규USP후보": "raw_usp_df",
    "시장별_신규USP": "market_new_usp",
    "시장별_USP_TOP": "market_usp_top",
    "제품분석": "product_analysis",
    "브랜드분석": "brand_analysis",
    "USP그래프": "usp_graph_data",
}


for sheet_name, variable_name in optional_tables.items():

    if variable_name in globals():

        value = globals()[variable_name]

        if isinstance(value, pd.DataFrame):

            excel_sheets[sheet_name] = value.copy()


# ------------------------------------------------------------
# 검수 관련 시트
# ------------------------------------------------------------

excel_sheets["검수_요약"] = review_summary.copy()

excel_sheets["검수_전체기사"] = article_check.copy()

excel_sheets["검수_날짜별"] = date_check.copy()

excel_sheets["검수_본문카테고리"] = (
    body_only_check.copy()
)

excel_sheets["검수_출시리뉴얼"] = (
    review_issue_type.copy()
)

excel_sheets["검수_제목파싱"] = (
    title_quality_check.copy()
)

excel_sheets["검수_브랜드이슈"] = (
    brand_issue_check_df.copy()
)

excel_sheets["검수_이슈묶음"] = (
    issue_group_check.copy()
)

excel_sheets["검수_필요기사"] = (
    review_needed.copy()
)


# ------------------------------------------------------------
# Excel 저장
# ------------------------------------------------------------

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    for sheet_name, df in excel_sheets.items():

        if not isinstance(df, pd.DataFrame):
            continue

        # Excel 시트명 최대 31자 제한
        sheet_name = str(sheet_name)[:31]

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# ------------------------------------------------------------
# Excel 서식 적용
# ------------------------------------------------------------

wb = load_workbook(output_file)


# 검수 관련 시트 표시용 색상
review_sheets = [
    "검수_요약",
    "검수_전체기사",
    "검수_날짜별",
    "검수_본문카테고리",
    "검수_출시리뉴얼",
    "검수_제목파싱",
    "검수_브랜드이슈",
    "검수_이슈묶음",
    "검수_필요기사"
]


for ws in wb.worksheets:

    # 첫 행 고정
    ws.freeze_panes = "A2"

    # 자동 필터
    if ws.max_row > 1 and ws.max_column > 0:

        ws.auto_filter.ref = ws.dimensions


    # 헤더 스타일
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


    # 열 너비 자동 조정
    for column_cells in ws.columns:

        max_length = 0

        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            try:
                value_length = len(
                    str(cell.value)
                )

                max_length = max(
                    max_length,
                    value_length
                )

            except:
                pass

        # 너무 넓어지지 않도록 제한
        width = min(
            max(max_length + 2, 10),
            60
        )

        ws.column_dimensions[
            column_letter
        ].width = width


    # 검수 시트는 제목/원문 때문에 너무 좁아지지 않도록
    if ws.title in review_sheets:

        for col in range(
            1,
            ws.max_column + 1
        ):

            header = ws.cell(
                row=1,
                column=col
            ).value

            if header in [
                "제목",
                "원문블록",
                "기사목록"
            ]:

                ws.column_dimensions[
                    get_column_letter(col)
                ].width = 60


    # 전체기사에서 URL이 있으면 하이퍼링크 처리
    if ws.title == "전체기사":

        headers = [
            ws.cell(
                row=1,
                column=c
            ).value
            for c in range(
                1,
                ws.max_column + 1
            )
        ]

        if "상품URL" in headers:

            url_col = (
                headers.index("상품URL") + 1
            )

            for row in range(
                2,
                ws.max_row + 1
            ):

                cell = ws.cell(
                    row=row,
                    column=url_col
                )

                if (
                    cell.value
                    and str(cell.value).startswith("http")
                ):

                    cell.hyperlink = str(
                        cell.value
                    )

                    cell.style = "Hyperlink"

            ws.column_dimensions[
                get_column_letter(url_col)
            ].width = 65


# ------------------------------------------------------------
# 검수 필요 기사 강조
# ------------------------------------------------------------

if "검수_필요기사" in wb.sheetnames:

    ws = wb["검수_필요기사"]

    # 제목/검수사유 열 찾기
    headers = {
        ws.cell(row=1, column=c).value: c
        for c in range(
            1,
            ws.max_column + 1
        )
    }

    # 검수사유 열 강조
    if "검수사유" in headers:

        col = headers["검수사유"]

        for row in range(
            2,
            ws.max_row + 1
        ):

            ws.cell(
                row=row,
                column=col
            ).font = Font(
                bold=True
            )


# ------------------------------------------------------------
# 저장 완료
# ------------------------------------------------------------

wb.save(output_file)


print("\n" + "=" * 80)
print("✅ Excel 저장 완료")
print("=" * 80)

print(f"파일명: {output_file}")
print(f"시트 수: {len(wb.sheetnames)}")

print("\n포함된 주요 검수 시트:")
for sheet in review_sheets:
    if sheet in wb.sheetnames:
        print(f"  ✓ {sheet}")


# ------------------------------------------------------------
# Colab 다운로드
# ------------------------------------------------------------

try:

    from google.colab import files

    files.download(output_file)

    print("\n⬇️ Excel 다운로드가 시작됩니다.")

except Exception:

    print(
        "\nColab 환경이 아니므로 자동 다운로드를 실행하지 않았습니다."
    )